<a href="https://colab.research.google.com/github/8parks/8parks/blob/main/ckks_friendly_RAG_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8_v1%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 환경 설정

## 기본 설정

In [ ]:
from google.colab import drive

# Drive 마운트
drive.mount('/content/drive')

# 작업 디렉토리 설정
import os
work_dir = '/content/drive/MyDrive/ckks_friendly_rag'
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

print(f"작업 디렉토리: {os.getcwd()}")

Mounted at /content/drive
작업 디렉토리: /content/drive/MyDrive/ckks_friendly_rag


Install

In [ ]:
!pip install -q sentence-transformers chromadb langchain langchain-community langchain_chroma
!pip install -q pypdf python-docx tqdm numpy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6

In [ ]:
# 암호화 Desilo
!pip install desilofhe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.2/930.2 kB 14.8 MB/s eta 0:00:00


Libararies

In [ ]:
from pathlib import Path
from typing import Iterator, Tuple
import os

# DB
import sqlite3
from langchain_chroma import Chroma

import desilofhe
from desilofhe import Engine

import numpy as np

Path Variables

In [ ]:
# 디렉토리 PATH
base_path = Path(work_dir)
nomic_dir = base_path / 'PoC_nomic-embed-v1.5 (1)'

KEY_PATH = base_path / "keys"
SECRET_KEY_PATH = KEY_PATH / "secret.key"
PUBLIC_KEY_PATH = KEY_PATH / "public.key"

# DB
db_path = nomic_dir/ 'encrypted.db'
col_db_path =nomic_dir / 't_col_encrypted.db'
chroma_path = base_path / 'chroma.sqlite3'

# Document
doc_path = base_path / "cat-facts.txt"

# Model
cache_dir = '/content/drive/MyDrive/RAG_Project/model_cache'

Model

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

# Drive에 모델 캐시
os.makedirs(cache_dir, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

""" nomic v1.5 """
model = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v1.5",
    trust_remote_code=True,
    device=device,
    cache_folder=cache_dir
)

""" nomic-embed-text-v2-moe
model = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v2-moe",
    trust_remote_code=True,
    device=device,
    cache_folder=cache_dir
)
"""

print(f"모델 로드 완료 ({device})")
print(f"차원: {model.get_sentence_embedding_dimension()}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/547M [00:00<?, ?B/s]

모델 로드 완료 (cpu)
차원: 768


Variables

In [ ]:
# Desilo fhe
engine = Engine(slot_count = 256)

In [ ]:
model_name = "nomic/"+"nomic-embed-text-v1.5"
slot_count = engine.slot_count
dim = model.get_sentence_embedding_dimension()

print(f"slot_count: {slot_count} \ndimension: {dim}")

slot_count: 256 
dimension: 768


dim이 768

## 사용 함수 정리

db 저장

In [ ]:
import sqlite3

def save_ciphertexts_sqlite(db_path: str, ids, cipher_bytes_list, table="encrypted_vectors"):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            id TEXT PRIMARY KEY,
            ciphertext BLOB
        )
    """)

    # executemany가 빠름
    cur.executemany(
        f"INSERT OR REPLACE INTO {table} (id, ciphertext) VALUES (?, ?)",
        [(doc_id, sqlite3.Binary(blob)) for doc_id, blob in zip(ids, cipher_bytes_list)]
    )

    conn.commit()
    conn.close()

암호화 DB(SQLite) 불러오기

In [ ]:
import sqlite3

def load_ciphertexts_sqlite(db_path: str, engine, table="encrypted_vectors"):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()

    cur.execute(f"SELECT id, ciphertext FROM {table} ORDER BY id")
    out = []

    for row in cur.fetchall():
        blob = row["ciphertext"]
        ct = engine.deserialize_ciphertext(blob)
        out.append(ct)

    conn.close()
    return out


암호화 로직

In [ ]:
def encrypt_values(vector_list, engine, public_key):
  encrypted_embeddings = []
  slot_count = engine.slot_count

  for vec in vector_list:
    packed = [0] * slot_count
    packed[0:dim] = vec[0:dim]
    ctx = engine.encrypt(packed, public_key)
    cipher_bytes = engine.serialize_ciphertext(ctx)
    encrypted_embeddings.append(cipher_bytes)

  return encrypted_embeddings

정규화

In [ ]:
import numpy as np

def l2_normalize(vec, eps=1e-12):
    vec = np.asarray(vec, dtype=np.float64)
    norm = np.linalg.norm(vec)
    return vec / (norm + eps)

Dot product

In [ ]:
def encrypted_inner_product(ct_query, ct_db, engine, rel_key, rot_key):
    ct_mul = engine.multiply(ct_query, ct_db, rel_key)

    n = engine.slot_count   # 또는 truncate_dim 256으로!
    dim2 = 1 << (n - 1).bit_length()

    step = 1
    while step < dim2:
        ct_rot = engine.rotate(ct_mul, rot_key, step)
        ct_mul = engine.add(ct_mul, ct_rot)
        step <<= 1

    return ct_mul

Top-k 연산

In [ ]:
def top_k_retrieval(results, k=10):
    sorted_results = sorted(results, key=lambda x: x["score"], reverse=True)
    return sorted_results[:k]


인덱스 재암호화

In [ ]:
def encrypt_indices(results, engine, public_key):
    encrypted_indices = []

    for res in results:
      encrypted_indices.append(engine.encrypt_ciphertext(res['id'], public_key))

    return encrypted_indices


행렬곱 연산

In [ ]:
def sum_first_n_slots(ct, n, engine, rot_key):
    """
    ct: Enc([x0, x1, ..., x_{n-1}, 0,0,...])
    return: Enc([S, S, S, ...])  (S = sum_{i=0}^{n-1} x_i)
    """
    # n을 다음 2의 거듭제곱으로 올려서 rotate-add 트리 구성
    dim2 = 1 << (n - 1).bit_length()
    acc = ct
    step = 1
    while step < dim2:
        acc = engine.add(acc, engine.rotate(acc, rot_key, step))
        step <<= 1
    return acc

def encrypted_dot(ct_a, ct_b, n, engine, rel_key, rot_key):
    """
    ct_a, ct_b: Enc vectors (packed), 길이 n까지 의미 있다고 가정.
    """
    ct_mul = engine.multiply(ct_a, ct_b, rel_key)
    ct_sum = sum_first_n_slots(ct_mul, n, engine, rot_key)
    return ct_sum  # 보통 모든 슬롯에 동일한 dot 값이 복제됨


In [ ]:
def encrypted_matmul(A_rows, B_cols, n, engine, rel_key, rot_key):
  """
  암호화된 행렬 간 곱셈을 해야함..
  K (k, 1024) * E (1024, n)
  C[r][c] = dot(A[r, :], B[:, c])
  """
  k = len(A_rows)
  m = len(B_cols)

  C = [[None] * m for _ in range(k)]
  for r in range(k):
      for c in range(m):
          C[r][c] = encrypted_dot(A_rows[r], B_cols[c], n, engine, rel_key, rot_key)
  return C

selector vector 만들기

In [ ]:
import numpy as np

def create_selector_matrix(i, k, n):
  mat = np.zeros((k, n), dtype = np.float64)

  mat[k,i] = 1.0

  return mat

In [ ]:
def create_selector_vector_row(i, n):
    vec = np.zeros(n, dtype=np.float64)
    vec[i] = 1.0
    return vec

In [ ]:
# row 기준으로 되어잇는 리스트를 col 기준으로 다시 묶음

def rows_to_cols(B_rows):
    # B_rows: [ [..dim..], [..dim..], ... ]  (D개)
    # return: [ [..D..], [..D..], ... ]      (dim개)
    return [list(col) for col in zip(*B_rows)]

In [ ]:
def create_selector_vector(target_indices, n):
    selectors = []
    for i in target_indices:
        selectors.append(create_selector_vector_row(i, n))

    enc_selectors = []

    for row in selectors:
      ct_row = engine.encrypt(row, public_key)
      # ct_row = row # 이 줄을 비활성화하여 암호화된 벡터를 사용하도록 함
      enc_selectors.append(ct_row)

    return enc_selectors

## Key

In [ ]:
# 키 파일 없으면 생성하지 말고 error 띄우도록 처리
os.makedirs(KEY_PATH, exist_ok=True)

if SECRET_KEY_PATH.exists():
    with open(SECRET_KEY_PATH, "rb") as f:
        secret_key = engine.deserialize_secret_key(f.read())
else:
    raise FileNotFoundError(
        f"Secret key not found at {SECRET_KEY_PATH.resolve()}.\n"
        "팀원이 준 키 파일을 확인하세요."
    )

if PUBLIC_KEY_PATH.exists():
    with open(PUBLIC_KEY_PATH, "rb") as f:
        public_key = engine.deserialize_public_key(f.read())
else:
     raise FileNotFoundError(
        f"Public key not found at {PUBLIC_KEY_PATH.resolve()}.\n"
        "팀원이 준 키 파일을 확인하세요."
    )

print(f"Engine slot count: {engine.slot_count}")
print(public_key)

Engine slot count: 256


In [ ]:
# 키 생성
REL_KEY_PATH = KEY_PATH / "relin.key"
ROT_KEY_PATH = KEY_PATH / "rotation.key"

if REL_KEY_PATH.exists():
    with open(REL_KEY_PATH, "rb") as f:
        rel_key = engine.deserialize_relinearization_key(f.read())
else:
     raise FileNotFoundError(
        f"Relin key not found at {REL_KEY_PATH.resolve()}.\n"
        "팀원이 준 키 파일을 확인하세요."
    )
if ROT_KEY_PATH.exists():
    with open(ROT_KEY_PATH, "rb") as f:
        rot_key = engine.deserialize_rotation_key(f.read())
else:
     raise FileNotFoundError(
        f"Rotation key not found at {ROT_KEY_PATH.resolve()}.\n"
        "팀원이 준 키 파일을 확인하세요."
    )

print(f"Relinearization key: {rel_key}")
print(f"Rotation key: {rot_key}")

Relinearization key: <desilofhe.RelinearizationKey object at 0x7e9004d88eb0>
Rotation key: <desilofhe.RotationKey object at 0x7e90085cebf0>


# DB 구축

### 임베딩

데이터 가공

In [ ]:
data = []

with open(doc_path, 'r', encoding="utf-8") as f:
    data = f.readlines()
    print(f"Loaded {len(data)}")

Loaded 150


In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(doc_path)
dataset = loader.load()

print(dataset)

[Document(metadata={'source': '/content/drive/MyDrive/ckks_friendly_rag/cat-facts.txt'}, page_content='On average, cats spend 2/3 of every day sleeping. That means a nine-year-old cat has been awake for only three years of its life.\nUnlike dogs, cats do not have a sweet tooth. Scientists believe this is due to a mutation in a key taste receptor.\nWhen a cat chases its prey, it keeps its head level. Dogs and humans bob their heads up and down.\nThe technical term for a cat’s hairball is a “bezoar.”\nA group of cats is called a “clowder.”\nFemale cats tend to be right pawed, while male cats are more often left pawed. Interestingly, while 90% of humans are right handed, the remaining 10% of lefties also tend to be male.\nA cat can’t climb head first down a tree because every claw on a cat’s paw points the same way. To get down from a tree, a cat must back down.\nCats make about 100 different sounds. Dogs make only about 10.\nA cat’s brain is biologically more similar to a human brain tha

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=0
)

docs = text_splitter.split_documents(dataset)
print(len(docs))

46


임베딩

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# SentenceTransformer 모델을 LangChain 임베딩 객체로 래핑
# trust_remote_code=True 추가, show_progress는 HuggingFaceEmbeddings의 파라미터로 직접 전달
langchain_embeddings = HuggingFaceEmbeddings(
    model_name="nomic-ai/nomic-embed-text-v1.5",
    model_kwargs={'device': device, 'trust_remote_code': True},
    show_progress=False # encode_kwargs 대신 여기에 show_progress를 직접 설정
)

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=langchain_embeddings,
    persist_directory=chroma_path,
    collection_name="nomic_embed"
)

vec_db = vectorstore.get(include=['embeddings', 'documents'])

print(f"vec_db의 타입: {type(vec_db)}")
if isinstance(vec_db, dict):
    print(f"vec_db의 키: {vec_db.keys()}")

/tmp/ipython-input-351/1516268537.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  langchain_embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

vec_db의 타입: <class 'dict'>
vec_db의 키: dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])


In [ ]:
print(vec_db['ids'])

['bb431323-15f7-4337-b2e7-a632b35c4232', '8416b38f-40ad-4004-8bb1-5651ec74a8ba', 'd304c079-22af-4145-9097-c16db0b175dc', 'dd500385-0829-4637-a5c7-e2dced10e5e9', 'f30f2ff1-3fca-40ad-923d-f0bb33052a33', 'e6d7720c-e6dc-4525-87ff-c7cc83622661', '81a57261-f599-4dd2-b48b-94d9f016a92d', '05626d00-de6e-4ac2-b66a-9a20fd844203', 'fdfe8ffa-0246-43ca-a21e-0d74211b20ed', '52aedb51-739c-4e06-bf7a-ffd70b1fc008', 'd47202fb-518c-4777-9480-7e61c99c484a', 'afb15d50-3fd6-42fc-9e70-070c75cbf498', '5d62aec1-125d-4547-a66b-8e3b9be671cf', '9e862f7a-2f05-4254-bc2b-e08b649c51e0', '1b7043d2-caff-4252-bb11-fd866fcee893', '0c7ca24a-7f13-4903-b212-22c337d8990e', '5408e4dd-ebfe-4a0c-96e1-a1de9f656559', '4af32463-635e-4223-9508-b4bda52da0db', '80e317b0-f93f-40bc-b04c-5b5675563303', 'c20e3faf-0162-4f2f-a41e-99937b055a64', '20b333f2-003c-4462-a8aa-729a1dea8e0a', '123dc445-39c3-42b8-804b-8a20ce0e2e8f', 'ff1fff36-7239-4f7c-8a3a-5d3a5651100a', 'ba8c08d4-21dd-4cad-8d29-8dc4695902a2', 'e01e2eb7-6e42-430f-8376-b069be1589b2',

### 정규화

In [ ]:
normalized_embeddings = [
    l2_normalize(emb) for emb in vec_db['embeddings']
]

### 암호화

In [ ]:
encrypted_embeddings = []

for emb in normalized_embeddings:
    packed = [0] * slot_count
    packed[0:dim] = emb[0:dim]
    ctx = engine.encrypt(packed, public_key)
    cipher_bytes = engine.serialize_ciphertext(ctx)
    encrypted_embeddings.append(cipher_bytes)
print('finished')

finished


#### 암호화된 데이터 저장(Sqlite DB)

In [ ]:
# 인덱스 배열 만들기
index_arr = [str(i) for i in range(len(encrypted_embeddings))]

In [ ]:
import os

# Ensure the parent directory for the database file exists
os.makedirs(db_path.parent, exist_ok=True)

save_ciphertexts_sqlite(db_path, index_arr, encrypted_embeddings)

# DB Load

#### 평문 임베딩

In [ ]:
persist_db = Chroma(
    persist_directory=chroma_path,
    embedding_function=model,
    collection_name="nomic_embed"
)

In [ ]:
vec_db = persist_db.get(include=['embeddings', 'documents'])

#### 암호화된 벡터
1. SQLite에서 ciphertext BLOB을 bytes로 꺼내고
2. engine.deserialize_ciphertext(bytes)로 ciphertext 객체로 복원

In [ ]:
# 테이블명 확인
con = sqlite3.connect(col_db_path)
cur = con.cursor()
table_names_results = cur.execute("select name from sqlite_master where type='table';").fetchall()
con.close()

if table_names_results:
    print(f"Found tables in {col_db_path}:")
    for row in table_names_results:
        print(f"- {row[0]}")
    table_name = table_names_results[0][0] # Assuming the first table found is the one we're interested in for 'table_name' variable
else:
    table_name = "encrypted_vectors"
    print(f"Warning: No tables found in the database at {col_db_path}. Assuming default table name '{table_name}'. Please verify the database creation.")

table_name

Found tables in /content/drive/MyDrive/ckks_friendly_rag/PoC_nomic-embed-v1.5 (1)/t_col_encrypted.db:
- encrypted_vectors


'encrypted_vectors'

In [ ]:
encrypted_db_cts = load_ciphertexts_sqlite(db_path, engine, 'encrypted_vectors')

In [ ]:
import os
print(col_db_path)                        # 경로 확인
print(os.path.exists(col_db_path))        # 파일 존재 여부

/content/drive/MyDrive/ckks_friendly_rag/PoC_nomic-embed-v1.5 (1)/t_col_encrypted.db
True


In [ ]:
import sqlite3
conn = sqlite3.connect(col_db_path)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(cursor.fetchall())  # 어떤 테이블이 있는지 출력
conn.close()

[('encrypted_vectors',)]


In [ ]:
import os
key_path = "/content/drive/MyDrive/ckks_friendly_rag/keys/secret.key"
print(os.path.exists(key_path))

True


In [ ]:
t_encrypted_db_cts = load_ciphertexts_sqlite(col_db_path, engine, 'encrypted_vectors')

In [ ]:
emb_array = np.vstack(t_encrypted_db_cts)
print(emb_array.shape)

(58, 1)


In [ ]:
print(t_encrypted_db_cts)

[<desilofhe.Ciphertext object at 0x7e9042aa6530>, <desilofhe.Ciphertext object at 0x7e8fc44bf6b0>, <desilofhe.Ciphertext object at 0x7e9042b125b0>, <desilofhe.Ciphertext object at 0x7e9042b107f0>, <desilofhe.Ciphertext object at 0x7e8fc79c03f0>, <desilofhe.Ciphertext object at 0x7e8fc4483970>, <desilofhe.Ciphertext object at 0x7e8fc4482db0>, <desilofhe.Ciphertext object at 0x7e8fc44837b0>, <desilofhe.Ciphertext object at 0x7e8fc44ddff0>, <desilofhe.Ciphertext object at 0x7e8fbff6b330>, <desilofhe.Ciphertext object at 0x7e8fc44dfe30>, <desilofhe.Ciphertext object at 0x7e8fc44dfdb0>, <desilofhe.Ciphertext object at 0x7e8fc44fb9f0>, <desilofhe.Ciphertext object at 0x7e8fc44f9570>, <desilofhe.Ciphertext object at 0x7e8fc44f8770>, <desilofhe.Ciphertext object at 0x7e9042af3730>, <desilofhe.Ciphertext object at 0x7e9042af0030>, <desilofhe.Ciphertext object at 0x7e9042976f70>, <desilofhe.Ciphertext object at 0x7e90429765f0>, <desilofhe.Ciphertext object at 0x7e9042976e70>, <desilofhe.Cipherte

In [ ]:
print(f"len(t_encrypted_db_cts) = {len(t_encrypted_db_cts)}")

len(t_encrypted_db_cts) = 58


# Retrieval

### Retrieval 단계 순서대로

쿼리

In [ ]:
#query = input("질문을 입력하세요: ")
query = "cat's food"

emb_query = model.encode(query, normalize_embeddings=True, truncate_dim=256)
#emb_query = l2_normalize(emb_query)
ct_q = engine.encrypt(emb_query, public_key)

쿼리와 벡터 db dot product

근데 여기에서 t가 아니라 다시 encrypted db_cts 쓰는 이유 궁금

In [ ]:
inner_products = []

for ct_db in encrypted_db_cts:
    ip = encrypted_inner_product(ct_q, ct_db, engine, rel_key, rot_key)
    inner_products.append(ip)

In [ ]:
index_arr = [str(i) for i in range(len(encrypted_db_cts))]

serialized_inner_products = [engine.serialize_ciphertext(ip) for ip in inner_products]

(인하우스) Sorting + Top-K

In [ ]:
# 임의로 인덱스 생성
indices = [i for i in range(len(inner_products))]

In [ ]:
result_scores = []
for i, ip in enumerate(inner_products):
    decrypted_score = engine.decrypt(ip, secret_key)[0]
    result_scores.append({"id": indices[i], "score": decrypted_score})

In [ ]:
top_k_results = top_k_retrieval(result_scores, k=10)
for res in top_k_results:
    print(f"ID: {res['id']}, Score: {res['score']:.6f}")

ID: 33, Score: 0.299717
ID: 84, Score: 0.299717
ID: 114, Score: 0.299717
ID: 165, Score: 0.299717
ID: 102, Score: 0.295443
ID: 21, Score: 0.295443
ID: 72, Score: 0.295443
ID: 153, Score: 0.295443
ID: 3, Score: 0.273599
ID: 162, Score: 0.273599


In [ ]:
import pandas as pd

# top_k_results를 DataFrame으로 변환
df_top_k = pd.DataFrame(top_k_results)

# CSV 파일 저장 경로 설정 (base_path 안에 'results' 디렉토리를 생성하고 저장)
results_dir = base_path / 'results'
os.makedirs(results_dir, exist_ok=True)
output_csv_path = results_dir / 'top_k_results.csv'

# DataFrame을 CSV 파일로 저장
df_top_k.to_csv(output_csv_path, index=False)

print(f"Top-K 결과를 다음 경로에 저장했습니다: {output_csv_path}")
print(df_top_k.head())

Top-K 결과를 다음 경로에 저장했습니다: /content/drive/MyDrive/ckks_friendly_rag/results/top_k_results.csv
    id     score
0   33  0.299717
1   84  0.299717
2  114  0.299717
3  165  0.299717
4  102  0.295443


In [ ]:
top_k_indices = [res['id'] for res in top_k_results]

top_k_indices

[33, 84, 114, 165, 102, 21, 72, 153, 3, 162]

In [ ]:
ids = [f"id{int(r['id'])+1}" for r in top_k_results]

plain = vectorstore.get(
    ids=ids,
    include=["documents", "embeddings"]
)

for i, doc in zip(plain["ids"], plain["documents"]):
    print(i, doc[:120])

In [ ]:
print("requested ids:", ids[:10], "... total", len(ids))
plain = vectorstore.get(ids=ids, include=["documents", "embeddings"])
print("returned keys:", plain.keys())
print("returned ids:", plain["ids"])
print("n_returned:", len(plain["ids"]))

requested ids: ['id34', 'id85', 'id115', 'id166', 'id103', 'id22', 'id73', 'id154', 'id4', 'id163'] ... total 10
returned keys: dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])
returned ids: []
n_returned: 0


In [ ]:
all_data = vectorstore.get(include=[])
print("total items in this collection:", len(all_data["ids"]))

total items in this collection: 184


In [ ]:
print("topk raw ids:", [r["id"] for r in top_k_results])
sample = vectorstore.get(include=["documents"])
print("sample ids:", sample["ids"][:30])
print("total:", len(vectorstore.get(include=[])["ids"]))

topk raw ids: [33, 84, 114, 165, 102, 21, 72, 153, 3, 162]
sample ids: ['bb431323-15f7-4337-b2e7-a632b35c4232', '8416b38f-40ad-4004-8bb1-5651ec74a8ba', 'd304c079-22af-4145-9097-c16db0b175dc', 'dd500385-0829-4637-a5c7-e2dced10e5e9', 'f30f2ff1-3fca-40ad-923d-f0bb33052a33', 'e6d7720c-e6dc-4525-87ff-c7cc83622661', '81a57261-f599-4dd2-b48b-94d9f016a92d', '05626d00-de6e-4ac2-b66a-9a20fd844203', 'fdfe8ffa-0246-43ca-a21e-0d74211b20ed', '52aedb51-739c-4e06-bf7a-ffd70b1fc008', 'd47202fb-518c-4777-9480-7e61c99c484a', 'afb15d50-3fd6-42fc-9e70-070c75cbf498', '5d62aec1-125d-4547-a66b-8e3b9be671cf', '9e862f7a-2f05-4254-bc2b-e08b649c51e0', '1b7043d2-caff-4252-bb11-fd866fcee893', '0c7ca24a-7f13-4903-b212-22c337d8990e', '5408e4dd-ebfe-4a0c-96e1-a1de9f656559', '4af32463-635e-4223-9508-b4bda52da0db', '80e317b0-f93f-40bc-b04c-5b5675563303', 'c20e3faf-0162-4f2f-a41e-99937b055a64', '20b333f2-003c-4462-a8aa-729a1dea8e0a', '123dc445-39c3-42b8-804b-8a20ce0e2e8f', 'ff1fff36-7239-4f7c-8a3a-5d3a5651100a', 'ba8c08d

In [ ]:
idx = top_k_results[0]["id"]  # 예: 33

# 1) 쿼리 평문 (256 맞춰)
emb_query = model.encode(query, normalize_embeddings=True)
q = np.array(emb_query[:engine.slot_count], dtype=np.float64)

# 2) 문서 평문 (vectorstore에서)
all_plain = vectorstore.get(include=["embeddings"])
doc_plain = np.array(all_plain["embeddings"][idx][:engine.slot_count], dtype=np.float64)

# 3) normalize 맞추기 (혹시 모르니까)
doc_plain = doc_plain / (np.linalg.norm(doc_plain) + 1e-12)

# 4) 평문 dot
plain_score = float(np.dot(q, doc_plain))

# 5) 암호화 score
enc_score = top_k_results[0]["score"]

print("plain_score:", plain_score)
print("enc_score:  ", enc_score)
print("diff:", abs(plain_score - enc_score))

plain_score: 0.3703699421243637
enc_score:   0.29971651677288347
diff: 0.07065342535148023


In [ ]:
all_plain = vectorstore.get(include=["documents", "embeddings"])  # ids, documents, embeddings 전부
print("total:", len(all_plain["ids"]))

for idx in [r["id"] for r in top_k_results]:
    chroma_id = all_plain["ids"][idx]
    doc = all_plain["documents"][idx]
    print(f"\nranked_idx={idx}  chroma_id={chroma_id}")
    print(doc[:200])

total: 184

ranked_idx=33  chroma_id=7bcae5c4-b947-4cf9-8295-e51fbffda28b
It is a common belief that cats are color blind. However, recent studies have shown that cats can see blue, green and red.
A large majority of white cats with blue eyes are deaf. White cats with only 

ranked_idx=84  chroma_id=b48ed21d-cc87-456f-ab81-001d1ab08618
Though rare, cats can contract canine heart worms.
The gene in cats that causes the orange coat color are sexed linked, and is on the X sex chromosome. This gene may display orange or black. Thus, as 

ranked_idx=114  chroma_id=65f9cadf-2c90-4817-a17c-645aee5cfd04
Cats have about 130,000 hairs per square inch (20,155 hairs per square centimeter). i
The heaviest cat on record is Himmy, a Tabby from Queensland, Australia. He weighed nearly 47 pounds (21 kg). He d

ranked_idx=165  chroma_id=ad29715a-5493-47e2-9875-f9c26a5e9f4e
Cats spend nearly 1/3 of their waking hours cleaning themselves.
Grown cats have 30 teeth. Kittens have about 26 temporary teeth, wh

In [ ]:
import numpy as np

idx = top_k_results[0]["id"]  # 예: 33
n = engine.slot_count         # 256

# --- 1) HE 문서 벡터(실제로 inner_product에 들어간 것) ---
doc_he = engine.decrypt(encrypted_db_cts[idx], secret_key)[:n]
doc_he = np.array(doc_he, dtype=np.float64)

# --- 2) vectorstore 문서 벡터(평문) ---
all_plain = vectorstore.get(include=["embeddings", "documents"])
doc_vs_768 = np.array(all_plain["embeddings"][idx], dtype=np.float64)
doc_vs = doc_vs_768[:n]

# 비교를 위해 둘 다 같은 방식으로 정규화(혹시 몰라서)
doc_he_n = doc_he / (np.linalg.norm(doc_he) + 1e-12)
doc_vs_n = doc_vs / (np.linalg.norm(doc_vs) + 1e-12)

cos = float(np.dot(doc_he_n, doc_vs_n))
max_abs = float(np.max(np.abs(doc_he_n - doc_vs_n)))

print("idx:", idx)
print("cos(doc_he, doc_vs):", cos)
print("max_abs_diff:", max_abs)
print("doc snippet:", all_plain["documents"][idx][:120] if all_plain["documents"] else "NO DOCS")

idx: 33
cos(doc_he, doc_vs): 0.7588830243500115
max_abs_diff: 0.14665394714587396
doc snippet: It is a common belief that cats are color blind. However, recent studies have shown that cats can see blue, green and re


In [ ]:
import numpy as np

idx = 33
n = engine.slot_count  # 256

# 실제로 사용한 query/doc 벡터를 복호해서 baseline 만들기
q_he = np.array(engine.decrypt(ct_q, secret_key)[:n], dtype=np.float64)
d_he = np.array(engine.decrypt(encrypted_db_cts[idx], secret_key)[:n], dtype=np.float64)

plain_from_he_inputs = float(np.dot(q_he, d_he))

he_score = float(engine.decrypt(inner_products[idx], secret_key)[0])

print("plain_from_he_inputs:", plain_from_he_inputs)
print("he_score:", he_score)
print("diff:", abs(plain_from_he_inputs - he_score))

plain_from_he_inputs: 0.29971651654989495
he_score: 0.29971651677288347
diff: 2.2298851654056762e-10


In [ ]:
import numpy as np

# 1) 평문 임베딩 (vectorstore embeddings 사용)
all_plain = vectorstore.get(include=["documents","embeddings"])
E = np.array(all_plain["embeddings"], dtype=np.float64)   # (N,768)
E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)

# 2) 쿼리 임베딩 (같은 모델/정규화)
q = model.encode("cat's food", normalize_embeddings=True)
q = np.array(q, dtype=np.float64)
q = q / (np.linalg.norm(q) + 1e-12)

# 3) cosine 점수
scores = E @ q
top = np.argsort(-scores)[:10]

for rank, idx in enumerate(top, 1):
    print(rank, idx, float(scores[idx]))
    print(all_plain["documents"][idx][:160].replace("\n"," ") )
    print()

1 25 0.7175530856865407
A cat’s nose pad is ridged with a unique pattern, just like the fingerprint of a human. If they have ample water, cats can tolerate temperatures up to 133 °F. F

2 71 0.7175530856865407
A cat’s nose pad is ridged with a unique pattern, just like the fingerprint of a human. If they have ample water, cats can tolerate temperatures up to 133 °F. F

3 117 0.7175530856865407
A cat’s nose pad is ridged with a unique pattern, just like the fingerprint of a human. If they have ample water, cats can tolerate temperatures up to 133 °F. F

4 163 0.7175530856865407
A cat’s nose pad is ridged with a unique pattern, just like the fingerprint of a human. If they have ample water, cats can tolerate temperatures up to 133 °F. F

5 82 0.7149922126294517
Cats must have fat in their diet because they can’t produce it on their own. Some common houseplants poisonous to cats include: English Ivy, iris, mistletoe, ph

6 174 0.7149922126294517
Cats must have fat in their diet because th

In [ ]:
from collections import Counter

all_plain = vectorstore.get(include=["documents"])
docs = [d.strip() for d in all_plain["documents"] if d]

cnt = Counter(docs)
dups = [(doc, c) for doc, c in cnt.items() if c > 1]
dups_sorted = sorted(dups, key=lambda x: -x[1])

print("total docs:", len(docs))
print("unique docs:", len(cnt))
print("duplicated groups:", len(dups_sorted))
print("top duplicate count:", dups_sorted[0][1] if dups_sorted else 1)

if dups_sorted:
    print("\nmost duplicated doc (first 200 chars):")
    print(dups_sorted[0][0][:200])

total docs: 184
unique docs: 46
duplicated groups: 46
top duplicate count: 4

most duplicated doc (first 200 chars):
On average, cats spend 2/3 of every day sleeping. That means a nine-year-old cat has been awake for only three years of its life.
Unlike dogs, cats do not have a sweet tooth. Scientists believe this i


암호화된 상태로 top-k 문서 찾아내기 (행렬곱 연산)

In [ ]:
idx_selector_vector = create_selector_vector(top_k_indices, slot_count)

In [ ]:
matmul_res = encrypted_matmul(idx_selector_vector, t_encrypted_db_cts, slot_count, engine, rel_key, rot_key)

In [ ]:
engine.decrypt(matmul_res[0][2], secret_key)

array([-0.06383516, -0.06383516, -0.06383516, -0.06383516, -0.06383516,
       -0.06383516, -0.06383516, -0.06383516, -0.06383516, -0.06383516,
       -0.06383516, -0.06383516, -0.06383516, -0.06383516, -0.06383516,
       -0.06383516, -0.06383516, -0.06383516, -0.06383516, -0.06383516,
       -0.06383516, -0.06383516, -0.06383516, -0.06383516, -0.06383516,
       -0.06383516, -0.06383516, -0.06383516, -0.06383516, -0.06383516,
       -0.06383516, -0.06383516, -0.06383516, -0.06383516, -0.06383516,
       -0.06383516, -0.06383516, -0.06383516, -0.06383516, -0.06383516,
       -0.06383516, -0.06383516, -0.06383516, -0.06383516, -0.06383516,
       -0.06383516, -0.06383516, -0.06383516, -0.06383516, -0.06383516,
       -0.06383516, -0.06383516, -0.06383516, -0.06383516, -0.06383516,
       -0.06383516, -0.06383516, -0.06383516, -0.06383516, -0.06383516,
       -0.06383516, -0.06383516, -0.06383516, -0.06383516, -0.06383516,
       -0.06383516, -0.06383516, -0.06383516, -0.06383516, -0.06

In [ ]:
res = []

for i in matmul_res:
  tmp = []
  for j in i:
    tmp.append(engine.decrypt(j, secret_key)[0])
  res.append(tmp)

In [ ]:
for r in res:
  print(r)

[np.float64(0.01032214655574933), np.float64(-0.015490716990088502), np.float64(-0.06383515931510936), np.float64(-0.04324905181699187), np.float64(-0.04229524366590431), np.float64(-0.026781599983362), np.float64(-0.02353979591232293), np.float64(-0.02573433013946341), np.float64(-0.04058617726096796), np.float64(-0.014950244208448935), np.float64(-0.05825091876445558), np.float64(-0.05478861537439187), np.float64(-0.0717974301372279), np.float64(-0.06201669137872269), np.float64(-0.03744851493641563), np.float64(-0.01648516349547135), np.float64(-0.039544352059295136), np.float64(0.005245258295795338), np.float64(-0.0485390008731813), np.float64(-0.017151184843439627), np.float64(0.04563420669959857), np.float64(-0.04422831879384312), np.float64(-0.008506578656395536), np.float64(-0.014417847271642173), np.float64(0.00832801085303902), np.float64(-0.07454629948798903), np.float64(-0.04270035736871408), np.float64(-0.03072282638834985), np.float64(-0.05690915848987261), np.float64(-0.

In [ ]:
res_np = np.array(res)
res_np.shape

(10, 58)

In [ ]:
print("N docs (encrypted_db_cts):", len(encrypted_db_cts))
print("len(t_encrypted_db_cts):", len(t_encrypted_db_cts))

# ciphertext 하나 복호해서 슬롯 길이 확인
v0 = engine.decrypt(t_encrypted_db_cts[0], secret_key)
print("one t_ct slots:", len(v0))
print("first 5:", v0[:5])

N docs (encrypted_db_cts): 184
len(t_encrypted_db_cts): 58
one t_ct slots: 256
first 5: [ 0.06602725  0.08912549 -0.2694017  -0.01914103  0.09858287]


디버깅 추가


In [ ]:
a = engine.decrypt(t_encrypted_db_cts[0], secret_key)
print(a[:20])  # 값이 어떻게 생겼는지
print(a)       # 256개 슬롯 전체

[ 0.06602725  0.08912549 -0.2694017  -0.01914103  0.09858287  0.08488719
 -0.11778896  0.05371458 -0.0352945  -0.00782371 -0.04036004  0.05820274
  0.12998889  0.05375615 -0.03970126 -0.09555845 -0.02678689 -0.09894831
  0.04440593  0.05783151]
[ 6.60272539e-02  8.91254917e-02 -2.69401699e-01 -1.91410296e-02
  9.85828713e-02  8.48871917e-02 -1.17788956e-01  5.37145846e-02
 -3.52944955e-02 -7.82371312e-03 -4.03600447e-02  5.82027398e-02
  1.29988894e-01  5.37561513e-02 -3.97012606e-02 -9.55584496e-02
 -2.67868880e-02 -9.89483073e-02  4.44059260e-02  5.78315072e-02
  1.36574656e-02  3.85471843e-02  2.86705121e-02 -4.54525016e-02
  1.32567612e-02  1.37266308e-01 -4.65777852e-02  3.35356710e-03
 -8.66776425e-03  1.28540739e-01  1.13215089e-01  1.09414114e-02
 -5.74703328e-03  1.03221461e-02 -4.48895842e-02 -1.04839103e-02
  4.13304046e-02  6.41243905e-02  8.06863010e-02  9.46552679e-02
  7.24692866e-02  4.06629071e-02  4.28734832e-02  1.17910840e-02
  6.41707610e-03 -3.65418494e-02  6.7455

In [ ]:
# encrypted_db_cts[0]과 t_encrypted_db_cts[0]을 복호화해서 비교
a = engine.decrypt(encrypted_db_cts[0], secret_key)
b = engine.decrypt(t_encrypted_db_cts[0], secret_key)
print(np.allclose(a, b))  # True면 동일한 데이터

ValueError: operands could not be broadcast together with shapes (768,) (256,) 

In [ ]:
import os, sqlite3

def inspect_db(path):
    print("\n===", path, "===")
    print("exists:", os.path.exists(path))
    if os.path.exists(path):
        print("size(bytes):", os.path.getsize(path))

    conn = sqlite3.connect(path)
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    print("tables:", cur.fetchall())

    # encrypted_vectors가 있는 경우만
    try:
        cur.execute("SELECT COUNT(*) FROM encrypted_vectors")
        print("encrypted_vectors row count:", cur.fetchone()[0])

        cur.execute("SELECT MIN(CAST(id as INT)), MAX(CAST(id as INT)) FROM encrypted_vectors")
        print("min/max id:", cur.fetchone())

        cur.execute("SELECT id, LENGTH(ciphertext) FROM encrypted_vectors ORDER BY CAST(id as INT) ASC LIMIT 3")
        print("first 3:", cur.fetchall())

        cur.execute("SELECT id, LENGTH(ciphertext) FROM encrypted_vectors ORDER BY CAST(id as INT) DESC LIMIT 3")
        print("last 3:", cur.fetchall())
    except Exception as e:
        print("encrypted_vectors inspect failed:", e)

    conn.close()

inspect_db(db_path)
inspect_db(col_db_path)


=== /content/drive/MyDrive/ckks_friendly_rag/PoC_nomic-embed-v1.5 (1)/encrypted.db ===
exists: True
size(bytes): 1303855104
tables: [('encrypted_vectors',)]
encrypted_vectors row count: 184
min/max id: (0, 183)
first 3: [('0', 7078298), ('1', 7078298), ('2', 7078298)]
last 3: [('183', 7078298), ('182', 7078298), ('181', 7078298)]

=== /content/drive/MyDrive/ckks_friendly_rag/PoC_nomic-embed-v1.5 (1)/t_col_encrypted.db ===
exists: True
size(bytes): 137089024
tables: [('encrypted_vectors',)]
encrypted_vectors row count: 58
min/max id: (0, 57)
first 3: [('0', 2359442), ('1', 2359442), ('2', 2359442)]
last 3: [('57', 2359442), ('56', 2359442), ('55', 2359442)]


In [ ]:
all_plain = vectorstore.get(include=["documents"])
print("chroma count:", len(all_plain["documents"]))

chroma count: 184
